# Bathymetry — shaded relief

Render a GEBCO 2020 subset of the Mid-Atlantic Ridge near the Azores as a
**hillshaded** relief map. Hillshading turns a flat depth grid into an
intuitive 3D-looking surface that makes the rift valley and transform faults
pop — exactly how a marine geologist would eyeball seafloor structure.

In [ ]:
import tempfile

import numpy as np
from matplotlib import colormaps
from matplotlib.colors import LinearSegmentedColormap
from pyramids.dataset import Dataset
from pyramids.plot import ColorBar, ColorScaling, DataStyle

from earthlens.core import EarthLens

In [ ]:
out = tempfile.mkdtemp()
paths = EarthLens(
    data_source='gebco',
    dataset='gebco_2020',
    aoi=[-29.5, 36.2, -27.7, 38.0],
    path=out,
).download(progress_bar=False)

source = Dataset.read_file(paths[0])

stats = source.stats(approx_ok=False)
print(
    'grid',
    source.shape,
    '| depth range',
    round(float(stats['min'].iloc[0])),
    '..',
    round(float(stats['max'].iloc[0])),
    'm',
)

## Hillshade

Light from the NW (`azdeg=315`) at 45 degrees elevation; `dx`/`dy` are the
approximate metres-per-pixel of a 15-arcsec grid, giving the hillshade a
physically correct horizontal scale. The vertical axis is a different story:
`vert_exag=30.0` stretches depth 30x relative to that horizontal scale. This
subset of the Mid-Atlantic Ridge is roughly 180 km across but only a few km
deep (see the printed depth range above), so a literal `vert_exag=1.0` renders
the seafloor as a nearly featureless plane — the 30x exaggeration is
what makes the rift valley and transform faults read as relief at all.

In [ ]:
deep_blues = LinearSegmentedColormap.from_list(
    'deep_blues', colormaps['Blues_r'](np.linspace(0.0, 0.82, 256))
)

source.plot(
    cmap=deep_blues,
    color=ColorScaling.power(gamma=0.7),
    colorbar=ColorBar(label='elevation (m relative to sea level)'),
    data_style=DataStyle(
        hillshade={
            'azimuth': 315,
            'altitude': 45,
            'vert_exag': 30.0,
            'dx': 460,
            'dy': 460,
            'blend_mode': 'soft',
        }
    ),
    title='GEBCO 2020 shaded relief — Mid-Atlantic Ridge (Azores)',
)